In [ ]:
# Download the repository
!git clone -q https://github.com/yasahi-hpc/JAX-PyTorch-Vlasov.git

# Make the source directory importable
import sys
sys.path.insert(0, "JAX-PyTorch-Vlasov/simulations/vlasov2d_2v/jax/src")

In [ ]:
!pip install "xarray[complete]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 177.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 189.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.5/377.5 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2

In [ ]:
import os
import jax.numpy as jnp
from vlasov2D2V import (
    run_vlp2d2v
)

nbiter, diag_steps = 10, 20
device = 'tpu'
out_dir = 'data_python'
physics_mode = True
lx, ly = 4.0*jnp.pi, 4.0*jnp.pi
Vx_max, Vy_max = 5.0, 5.0
dt = 0.05
epsilon = 0.001
device_name = 'TPUv5' if device == 'tpu' else 'cpu'
problems = [(32, 32), (32, 64), (64, 64), (64, 128), (128, 128)]

for dtype in ["float32"]:
    for (nx, nvx) in problems:
        for solver in range(3):
            if (solver != 2 and nx >= 64): continue
            run_vlp2d2v(
                nx=nx,
                ny=nx,
                nvx=nvx,
                nvy=nvx,
                lx=lx,
                ly=ly,
                Vx_max=Vx_max,
                Vy_max=Vy_max,
                nbiter=nbiter,
                diag_steps=diag_steps,
                dt=dt,
                out_dir=out_dir,
                physics_mode=physics_mode,
                epsilon=epsilon,
                solver_type=solver,
                dtype=dtype
            )

            # Rename the resulting file to the requested format
            old_filename = f'vlp2d_2v_{dtype}.txt'
            new_filename = f'vlp2d_2v_{device_name}_{dtype}_Nx{nx}_Nvx{nvx}_solver{solver}.txt'
            if os.path.exists(old_filename):
                os.rename(old_filename, new_filename)
                print(f'Renamed {old_filename} to {new_filename}')


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx32_Nvx32_solver0.txt
Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx32_Nvx32_solver1.txt
Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx32_Nvx32_solver2.txt
Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx32_Nvx64_solver0.txt
Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx32_Nvx64_solver1.txt
Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx32_Nvx64_solver2.txt
Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx64_Nvx64_solver2.txt
Renamed vlp2d_2v_float32.txt to vlp2d_2v_TPUv5_float32_Nx64_Nvx128_solver2.txt


In [ ]:
import zipfile
import os
from google.colab import files

# Define the name of the output zip file
zip_filename = 'vlp2d-2v_results.zip'

# Mapping of local directories to their desired names inside the zip
dump_mapping = {
    'jaxpr_dump': f'vlp2d_2v_{device_name}_{dtype}_jaxpr_dump',
    'hlo_dump': f'vlp2d_2v_{device_name}_{dtype}_hlo_dump'
}


# Create a zip archive
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Add .txt files from the current directory
    txt_files = [f for f in os.listdir('.') if f.startswith('vlp2d_2v_') and f.endswith('.txt')]
    for file in txt_files:
        zipf.write(file)

    # Add files from dump folders into renamed subdirectories
    for local_folder, zip_folder in dump_mapping.items():
        if os.path.exists(local_folder):
            for root, dirs, files_in_dir in os.walk(local_folder):
                for file in files_in_dir:
                    file_path = os.path.join(root, file)
                    # Construct the internal path: new_folder_name/filename
                    archive_path = os.path.join(zip_folder, file)
                    zipf.write(file_path, arcname=archive_path)

# Download the zip file
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>